In [ ]:
from PIL import Image
img = Image.open("sample1.png")

In [15]:
#converting password to key
def password_to_key(password):
    key=0
    for char in password:
        key+=ord(char)
    return key % 256

print(password_to_key("abc"))

38


In [16]:
#encrypting the message using XOR operation with a key
def encrypt_message(message,key):
    encrypted_message=[]
    for  char in message:
        encrypted=ord(char)^key
        encrypted_message.append(encrypted)
    message_len=len(encrypted_message)
    return encrypted_message,message_len
        

In [17]:
#converitng encrypted message to binary
def to_binary(encrypted_message):
    binary=""
    for num in encrypted_message:
        binary+=format(num,'08b')
    return binary

    


In [18]:
#chekking the binary representation of the encrypted message
message="I am handsome"
key=password_to_key("abc")
encrypted_message,message_len=encrypt_message(message,key)
binary_message=to_binary(encrypted_message)
print(encrypted_message)
print(binary_message)
print(message_len)

[111, 6, 71, 75, 6, 78, 71, 72, 66, 85, 73, 75, 67]
01101111000001100100011101001011000001100100111001000111010010000100001001010101010010010100101101000011
13


In [ ]:
#adding the binary message to the least significant bits of the image pixels
def add_to_Lsb(img,binary_message):
    pixels=img.load()

    bit_index=0
    for y in range(img.height):
        for x in range(img.width):
            if bit_index>=len(binary_message):
                break
            r,g,b=pixels[x,y]
            if bit_index<len(binary_message):
                r=(r & ~1) | int(binary_message[bit_index]) #clearing and adding message bit
                bit_index+=1
            if bit_index<len(binary_message):
                g=(g & ~1) | int(binary_message[bit_index]) 
                bit_index+=1     
            if bit_index<len(binary_message):
                b=(b & ~1) | int(binary_message[bit_index])
                bit_index+=1    
            pixels[x,y]=(r,g,b)
        if bit_index>=len(binary_message):
            break
add_to_Lsb(img,binary_message)
img.save("encoded_image.png")

In [25]:
#extracing the message from the image
enc_img=Image.open("encoded_image.png")
def extract_message(enc_img,message_len):
    binary_message=""
    encrypted_message=[]
    for y in range(enc_img.height):
        for x in range(enc_img.width):
            r,g,b=enc_img.getpixel((x,y))
            binary_message+=str(r & 1) #extracting the least significant bit
            binary_message+=str(g & 1)
            binary_message+=str(b & 1)
            while len(binary_message)>=8:
                byte=binary_message[:8]
                binary_message=binary_message[8:]
                
                encrypted_message.append(int(byte,2))

                #stop when the message length is reached
                if len(encrypted_message)>=message_len:
                    return encrypted_message
    return encrypted_message
encrypted_mesg=extract_message(enc_img,message_len)

In [ ]:
#decrypting the message using XOR operation with a passord-derived key
def decrypt_message(encrypted_message, key):
    message = ""
    for num in encrypted_message:
        char = chr(num ^ key)
        message += char
    return message 


decrypted_mesg=decrypt_message(encrypted_mesg,key)

In [ ]:
#printing the original and decrypted messages
print("Original Message:", message)
print("Decrypted Message:", decrypted_mesg)

Original Message: I am handsome
Decrypted Message: I am handsome
